## 🎯 Learning Objectives
* Apply K-Means clustering for customer segmentation.
* Understand the importance of feature scaling in distance-based algorithms.
* Utilize methods like the Elbow Method and Silhouette Score to determine the optimal number of clusters.
* Visualize clustering results and interpret the characteristics of different customer segments.


## Exercise: Customer Segmentation with K-Means Clustering

### Task Overview

In the dynamic world of business, understanding your customers is paramount for effective marketing and product development. Customer segmentation allows companies to divide their customer base into distinct groups based on shared characteristics, enabling highly targeted strategies. In this exercise, you will act as a Data Scientist tasked with segmenting a synthetic customer dataset using the K-Means clustering algorithm.

Your goal is to identify meaningful customer segments that a marketing team can use to tailor campaigns, personalize offers, and improve customer satisfaction.

### Dataset Description

You will be provided with a synthetic dataset containing customer information, including features like `Annual Income (k$)` and `Spending Score (1-100)`. Your task is to cluster customers based on these features.

### Requirements

1.  **Data Loading and Inspection**: Load the provided synthetic dataset and perform an initial inspection (e.g., `head()`, `info()`, `describe()`).
2.  **Feature Selection**: Identify the relevant features for clustering. For this exercise, focus on `Annual Income (k$)` and `Spending Score (1-100)`.
3.  **Feature Scaling**: Apply appropriate feature scaling to the selected features. Explain why scaling is necessary for K-Means.
4.  **Optimal Number of Clusters (K)**: Determine the optimal number of clusters (`k`) using at least two methods:
    *   **Elbow Method**: Plot the Within-Cluster Sum of Squares (WCSS) for a range of `k` values.
    *   **Silhouette Score**: Calculate and plot the Silhouette Score for a range of `k` values.
    *   Justify your chosen `k` based on these methods.
5.  **K-Means Clustering**: Apply the K-Means algorithm with your chosen optimal `k`.
6.  **Cluster Assignment**: Assign the cluster labels back to your original dataset.
7.  **Visualization**: Create a scatter plot to visualize the clusters. Use `Annual Income (k$)` on one axis and `Spending Score (1-100)` on the other, coloring points by their assigned cluster.
8.  **Cluster Interpretation**: Analyze the characteristics of each identified cluster. Describe what each segment represents in terms of income and spending behavior.

### Evaluation Criteria

*   **Correctness**: Accurate implementation of K-Means, scaling, and evaluation metrics.
*   **Clarity**: Well-commented code and clear explanations for each step.
*   **Insightfulness**: A thoughtful justification for the chosen `k` and a clear, actionable interpretation of the customer segments.
*   **Visualization Quality**: Clear and informative plots that effectively communicate the clustering results.

Good luck! Your insights will help the marketing team make data-driven decisions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set a random seed for reproducibility
np.random.seed(42)

# --- Generate Synthetic Customer Dataset ---
# We'll create a dataset that mimics typical customer segmentation scenarios
# with distinct groups based on income and spending habits.

num_customers = 500

# Define characteristics for different customer segments
# Segment 1: Low Income, Low Spending (Frugal)
income_1 = np.random.normal(loc=25, scale=5, size=int(num_customers * 0.2))
spending_1 = np.random.normal(loc=20, scale=5, size=int(num_customers * 0.2))

# Segment 2: Medium Income, Medium Spending (Average)
income_2 = np.random.normal(loc=55, scale=10, size=int(num_customers * 0.3))
spending_2 = np.random.normal(loc=50, scale=10, size=int(num_customers * 0.3))

# Segment 3: High Income, High Spending (Target/Premium)
income_3 = np.random.normal(loc=85, scale=15, size=int(num_customers * 0.2))
spending_3 = np.random.normal(loc=80, scale=15, size=int(num_customers * 0.2))

# Segment 4: Low Income, High Spending (Impulsive/New)
income_4 = np.random.normal(loc=30, scale=7, size=int(num_customers * 0.15))
spending_4 = np.random.normal(loc=75, scale=10, size=int(num_customers * 0.15))

# Segment 5: High Income, Low Spending (Careful/Established)
income_5 = np.random.normal(loc=90, scale=10, size=int(num_customers * 0.15))
spending_5 = np.random.normal(loc=25, scale=7, size=int(num_customers * 0.15))

# Combine all segments
annual_income = np.concatenate([income_1, income_2, income_3, income_4, income_5])
spending_score = np.concatenate([spending_1, spending_2, spending_3, spending_4, spending_5])

# Ensure spending score is within 1-100 range and income is positive
spending_score = np.clip(spending_score, 1, 100)
annual_income = np.clip(annual_income, 10, 150) # Clip income to a reasonable range

# Create DataFrame
data = pd.DataFrame({
    'CustomerID': range(1, num_customers + 1),
    'Gender': np.random.choice(['Male', 'Female'], size=num_customers),
    'Age': np.random.randint(18, 70, size=num_customers),
    'Annual Income (k$)': annual_income,
    'Spending Score (1-100)': spending_score
})

# Shuffle the DataFrame to mix segments
df = data.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- Synthetic Customer Dataset Generated ---")
print(df.head())
print("\n--- Dataset Info ---")
df.info()
print("\n--- Dataset Description ---")
df.describe()

# Visualize the raw data to get an initial feel for potential clusters
plt.figure(figsize=(10, 7))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', data=df, alpha=0.7)
plt.title('Raw Customer Data: Annual Income vs. Spending Score')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


### Your Implementation

Now it's your turn to implement the customer segmentation solution. Follow the requirements outlined in the task description. 

Remember to:
1.  Select the relevant features.
2.  Scale your data.
3.  Determine the optimal `k` using both the Elbow Method and Silhouette Score.
4.  Apply K-Means.
5.  Visualize and interpret your clusters.

Write your code in the cell below.


In [ ]:
### Reference Solution: Customer Segmentation with K-Means

# 1. Feature Selection
# We select 'Annual Income (k$)' and 'Spending Score (1-100)' as per the exercise description.
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features]

print("Selected features for clustering:")
print(X.head())
print("\n")

# 2. Feature Scaling
# K-Means is a distance-based algorithm, meaning it calculates distances between data points.
# If features have different scales (e.g., income in thousands vs. spending score 1-100),
# features with larger values will disproportionately influence the distance calculations.
# StandardScaler transforms the data to have a mean of 0 and a standard deviation of 1,
# ensuring all features contribute equally to the distance metric.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled features (first 5 rows):\n", X_scaled[:5])
print("\n")

# 3. Determine Optimal Number of Clusters (K)

# --- Method 1: Elbow Method (WCSS - Within-Cluster Sum of Squares) ---
# WCSS measures the sum of squared distances between each point and the centroid of its assigned cluster.
# As K increases, WCSS generally decreases. The 'elbow' point on the plot indicates where the decrease
# in WCSS starts to slow down significantly, suggesting an optimal K.
wcss = []
max_k = 10 # We'll check K from 1 to max_k-1
for i in range(1, max_k):
    # Initialize KMeans with 'k-means++' for smart initialization to avoid poor local optima
    # 'n_init=10' runs the algorithm 10 times with different centroid seeds and chooses the best result
    # 'random_state' ensures reproducibility
    kmeans = KMeans(n_clusters=i, init='k-means++', n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_) # inertia_ is the WCSS attribute

plt.figure(figsize=(10, 6))
plt.plot(range(1, max_k), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.xticks(range(1, max_k))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# --- Method 2: Silhouette Score ---
# The Silhouette Score measures how similar an object is to its own cluster compared to other clusters.
# It ranges from -1 to 1, where:
#   1: Means clusters are well-separated.
#   0: Means clusters are indifferent or overlapping.
#  -1: Means data points might be assigned to the wrong clusters.
# A higher silhouette score generally indicates better-defined clusters.
silhouette_scores = []
# Silhouette score is not defined for k=1, so we start from 2
for i in range(2, max_k):
    kmeans = KMeans(n_clusters=i, init='k-means++', n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(score)

plt.figure(figsize=(10, 6))
plt.plot(range(2, max_k), silhouette_scores, marker='o', linestyle='--')
plt.title('Silhouette Score for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(range(2, max_k))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# --- Justification for Optimal K ---
# Based on the Elbow Method plot, we observe a significant 'elbow' around K=5.
# The WCSS decreases sharply until K=5, after which the rate of decrease slows down considerably.
# For the Silhouette Score, K=5 also shows a relatively high score, indicating well-separated clusters.
# Therefore, we choose K=5 as the optimal number of clusters.
optimal_k = 5
print(f"\nChosen Optimal K: {optimal_k}\n")

# 4. K-Means Clustering with Optimal K
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

# 5. Cluster Assignment
df['Cluster'] = cluster_labels

print("DataFrame with assigned clusters (first 5 rows):")
print(df.head())
print("\n")

# 6. Visualization of Clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', data=df,
                palette='viridis', s=100, alpha=0.8, edgecolor='w')
plt.scatter(scaler.inverse_transform(kmeans.cluster_centers_)[:, 0], # Inverse transform centroids for original scale
            scaler.inverse_transform(kmeans.cluster_centers_)[:, 1], 
            s=300, c='red', marker='X', label='Centroids', edgecolor='black')
plt.title(f'Customer Segments (K={optimal_k})')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 7. Cluster Interpretation
# Let's examine the characteristics of each cluster by looking at the mean values
# of 'Annual Income' and 'Spending Score' for each segment.

cluster_summary = df.groupby('Cluster')[features].mean().round(2)
print("--- Cluster Summary (Mean Income and Spending Score) ---")
print(cluster_summary)
print("\n")

# Detailed interpretation of each cluster:
print("--- Interpretation of Customer Segments ---")

# Based on the synthetic data generation and typical segmentation patterns, 
# and observing the cluster summary and visualization:

# Cluster 0: (e.g., Low Income, Low Spending)
#   - Characteristics: Customers in this segment have relatively low annual income and low spending scores.
#   - Marketing Strategy: Price-sensitive offers, essential products, focus on value.

# Cluster 1: (e.g., Medium Income, Medium Spending)
#   - Characteristics: This is the 'average' customer segment, with moderate income and spending habits.
#   - Marketing Strategy: Standard promotions, loyalty programs, broad product range.

# Cluster 2: (e.g., High Income, High Spending)
#   - Characteristics: Affluent customers with high annual income and high spending scores. These are often the most valuable customers.
#   - Marketing Strategy: Premium products, exclusive offers, personalized luxury experiences, VIP programs.

# Cluster 3: (e.g., Low Income, High Spending)
#   - Characteristics: Customers with lower income but surprisingly high spending scores. This could indicate impulsive buyers, new customers trying out products, or those prioritizing certain categories.
#   - Marketing Strategy: Installment plans, attractive financing options, focus on trending products, upsell opportunities.

# Cluster 4: (e.g., High Income, Low Spending)
#   - Characteristics: High-income individuals who are cautious spenders. They might be looking for specific high-quality items or are very price-conscious despite their income.
#   - Marketing Strategy: Focus on product quality, durability, long-term value, targeted discounts on high-end items, building trust.

print("The specific labels (0, 1, 2, etc.) are arbitrary and depend on K-Means initialization.")
print("The key is to understand the *characteristics* of each group based on their feature means.")
print("For example, if Cluster 0 has mean income ~25k and spending ~20, it's the 'Frugal' segment.")
print("If Cluster 2 has mean income ~85k and spending ~80, it's the 'Premium' segment.")
